7. Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow
Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one
Cyntexa should use for a file source that arrives unpredictably throughout the day.

->
### 1. Batch CTAS
CTAS is mainly used for batch processing, especially for scheduled batches.
The cost is generally low, but latency can be high because data is processed in batches.

### 2. COPY INTO
COPY INTO can be used for batch as well as incremental file ingestion.
It is suitable when files arrive periodically. Cost and latency depend on how frequently we run the command.

### 3. Auto Loader
Auto Loader is best for unpredictable file arrivals.
It provides low latency and can incrementally process new files while keeping track of files that have already been processed.
It is suitable for continuously arriving files and can handle unpredictable file arrivals efficiently.

### 4. Lakeflow Declarative Pipelines
Lakeflow Declarative Pipelines provides a higher-level and managed way to define and operate data pipelines.
It simplifies data flow, dependencies, orchestration, and monitoring compared with managing multiple notebooks or scripts manually.
The latency depends on how the pipeline is configured, such as triggered or continuous execution.

### Recommendation

For a file source that arrives unpredictably throughout the day, **Auto Loader is the best choice**.

The main reasons are:
- It supports incremental/continuous file ingestion.
- It can handle files arriving at unpredictable times.
- It provides low latency.
- It keeps track of processed files.
- It is more suitable for continuously growing file sources than scheduled batch ingestion.

8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact
commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.
-->

### Scenario

Suppose a bad file comes at **2 AM** and Auto Loader loads this bad data into our Silver table:

`dev.autoloader.sales_autoloader`

Now the Silver table has corrupted data.

As a data engineer we need to find the last correct version and recover the table.

### Step 1: Check the table history

First, check the Delta table history to see when the bad data was added.

```sql
DESCRIBE HISTORY dev.autoloader.sales_autoloader;
```

### Step 2: Check the old version using Time Travel

```sql 
SELECT *
FROM dev.autoloader.sales_autoloader
VERSION AS OF 10;
```
### Step 3: Restore the table and validate

```sql
RESTORE TABLE dev.autoloader.sales_autoloader TO VERSION AS OF version_num;

DESCRIBE HISTORY dev.autoloader.sales_autoloader;
```

### If using pyspark and restoring using the overwrite the data 

```pyspark
df = (
    spark.read
    .format("delta")
    .option("versionAsOf", 10)
    .table("dev.autoloader.sales_autoloader")
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dev.autoloader.sales_autoloader")
)
```





9. (Data Analyst) Using DESCRIBE HISTORY, produce a 'data freshness' report showing how frequently a
given table is actually updated, to validate an SLA claim made to a business stakeholder.


Data freshness means **how frequently our table is getting updated**.

In this question, the business stakeholder has given an SLA, for example:

**"The table should be updated every 1 hour."**

As a Data Analyst, we need to verify whether this SLA is actually being followed.

First, we use `DESCRIBE HISTORY` to get the table history. It gives us information like the **version, timestamp, and operation**.

Then we can read this history into a DataFrame and select only the required columns.

After that, we use the previous update timestamp to calculate the **time gap between two updates**.

For example:

```text
09:00 → 10:00 = 60 minutes
10:00 → 11:05 = 65 minutes
11:05 → 12:00 = 55 minutes

In [0]:
%python 
from pyspark.sql.functions import *
from pyspark.sql.window import Window

history_df = spark.sql("""
    DESCRIBE HISTORY dev.autoloader.sales_autoloader""")

freshness_df = (
    history_df
    .select(
        "version",
        "timestamp",
        "operation"
    )
    .orderBy("timestamp")
)

freshness_df = freshness_df.withColumn(
    "previous_update",
    lag("timestamp").over(
        Window.orderBy("timestamp")
    )
)
freshness_df = freshness_df.withColumn(
    "time_gap_minutes",
    (
        unix_timestamp("timestamp")
        - unix_timestamp("previous_update")
    ) / 60
)

display(freshness_df)